# CareBot NLP intent classifier
Reproducible training and evaluation notebook for CareBridge. The classifier routes resource and safety questions; it does not diagnose or replace emergency services.

## Objectives
Classify user messages into curated intents, reject low-confidence queries, preserve deterministic emergency rules, and connect bed/blood intents to current API inventory.

In [ ]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from train_chatbot import load_training_data, build_pipeline, train, MODEL, METRICS


## Load curated training utterances
Training uses the version-controlled CareBot dataset. Three polite-prefix variants improve robustness without pretending to be new human-labelled observations.

In [ ]:
texts, labels, intents = load_training_data()
frame = pd.DataFrame({'text':texts,'intent':labels})
print(f'{len(frame)} examples across {frame.intent.nunique()} intents')
frame.sample(10,random_state=42)


In [ ]:
counts=frame.intent.value_counts().sort_values()
counts.plot.barh(figsize=(8,7),title='Augmented examples per intent',color='#0b7a70');plt.tight_layout()


## Quality gates

In [ ]:
assert frame.text.str.strip().ne('').all()
assert frame.intent.notna().all()
assert counts.min() >= 6
duplicates=frame.groupby('text').intent.nunique()
assert (duplicates<=1).all(), 'Conflicting labels found'


## Model design
Word TF-IDF captures phrases such as ‘ICU bed’; character TF-IDF tolerates spelling mistakes. Balanced logistic regression provides calibrated-enough probabilities for conservative rejection.

In [ ]:
model=build_pipeline()
model


## Stratified cross-validation
Cross-validation measures intent routing on held-out utterances. Augmented variants share wording, so this score is an engineering regression metric, not proof of real-user performance.

In [ ]:
evaluation_texts=[u for item in intents for u in item['utterances']]
evaluation_labels=[item['intent'] for item in intents for _ in item['utterances']]
evaluation_counts=pd.Series(evaluation_labels).value_counts()
folds=min(5,int(evaluation_counts.min()))
cv=StratifiedKFold(folds,shuffle=True,random_state=42)
scores=cross_val_score(model,evaluation_texts,evaluation_labels,cv=cv,scoring='accuracy')
pd.Series(scores,name='accuracy').describe()


In [ ]:
predicted=cross_val_predict(model,evaluation_texts,evaluation_labels,cv=cv)
report=pd.DataFrame(classification_report(evaluation_labels,predicted,output_dict=True,zero_division=0)).T
report.sort_values('f1-score').head(12)


## Train and export the API artifact

In [ ]:
metrics=train()
metrics


In [ ]:
artifact=joblib.load(MODEL)
assert artifact['metrics']['intents']==frame.intent.nunique()
assert artifact['metrics']['suitable_for_medical_diagnosis'] is False
json.loads(METRICS.read_text())


## Behaviour probes

In [ ]:
probes=['need o negative blood urgently','find an icu in jaipur','how is occupancy predicted','someone has chest pain','hello carebot','how can a hospital register']
prob=artifact['model'].predict_proba(probes)
pd.DataFrame({'message':probes,'intent':artifact['model'].predict(probes),'confidence':prob.max(axis=1).round(3)})


## Safety and deployment
Emergency phrases are intercepted before ML inference and always instruct the user to call 112. Predictions below the API threshold are rejected. Resource answers are enriched from database inventory and explicitly require direct hospital confirmation. Monitor unknown-rate, intent drift and incorrect emergency routing before expanding the dataset.